# Lesson 19: Deep Learning Architectures

*ML Academy — companion notebook*

Training deep nets, CNNs for images, RNNs for sequences, and Transformers.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-19.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 9. In Python

In [1]:
import numpy as np

def conv2d(image, kernel):
    kh, kw = kernel.shape
    out = np.zeros((image.shape[0] - kh + 1, image.shape[1] - kw + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = np.sum(image[i:i+kh, j:j+kw] * kernel)
    return out

image = np.array([[0, 0, 0, 9, 9, 9]] * 4, dtype=float)
edge = np.array([[-1, 0, 1]] * 3, dtype=float)
print(conv2d(image, edge))

fmap = np.array([[1, 3, 2, 0], [4, 6, 1, 1], [0, 2, 5, 7], [1, 1, 3, 2]], dtype=float)
print(fmap.reshape(2, 2, 2, 2).max(axis=(1, 3)))      # 2x2 max pooling

def attention(Q, K, V):
    scores = Q @ K.T / np.sqrt(K.shape[1])
    weights = np.exp(scores) / np.exp(scores).sum(axis=1, keepdims=True)   # softmax per row
    return weights, weights @ V

Q = K = np.array([[1, 0], [0, 1], [1, 1]], dtype=float)
V = np.array([[1, 0], [0, 1], [0.5, 0.5]])
weights, out = attention(Q, K, V)
print(weights.round(3))
print(out.round(3))

[[ 0. 27. 27.  0.]
 [ 0. 27. 27.  0.]]
[[6. 2.]
 [2. 7.]]
[[0.401 0.198 0.401]
 [0.198 0.401 0.401]
 [0.248 0.248 0.503]]
[[0.602 0.398]
 [0.398 0.602]
 [0.5   0.5  ]]


The PyTorch example below needs PyTorch installed (see pytorch.org), so it's shown as text rather than a runnable cell:

```python
import torch.nn as nn

model = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),   # 28x28x1  -> 28x28x16
    nn.ReLU(),
    nn.MaxPool2d(2),                              # -> 14x14x16
    nn.Conv2d(16, 32, kernel_size=3, padding=1),  # -> 14x14x32
    nn.ReLU(),
    nn.MaxPool2d(2),                              # -> 7x7x32
    nn.Flatten(),                                 # -> 1568 numbers
    nn.Dropout(0.25),
    nn.Linear(32 * 7 * 7, 10),                    # one score per digit
)
```

## 12. Going deeper: attention and residual connections, from scratch

In [2]:
import numpy as np

# Scaled dot-product attention from scratch, for a "sentence" of 4 tokens with 8-number vectors
rng = np.random.default_rng(0)
X = rng.normal(size=(4, 8))                                    # one row per token
Wq, Wk, Wv = (rng.normal(0, 1 / np.sqrt(8), size=(8, 8)) for _ in range(3))
Q, K, V = X @ Wq, X @ Wk, X @ Wv

def softmax(s):
    e = np.exp(s - s.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

A = softmax(Q @ K.T / np.sqrt(8))                              # attention weights, one row per token
out = A @ V                                                    # each output mixes the value vectors
print("attention weights (row i: how much token i looks at each token):")
print(A.round(2))
print("each row sums to:", A.sum(axis=1).round(3), "  output shape:", out.shape)

attention weights (row i: how much token i looks at each token):
[[0.34 0.13 0.33 0.2 ]
 [0.78 0.   0.05 0.17]
 [0.25 0.28 0.29 0.18]
 [0.22 0.28 0.21 0.29]]
each row sums to: [1. 1. 1. 1.]   output shape: (4, 8)


In [3]:
# Why divide by sqrt(d)? Dot products of random vectors grow with d, making softmax nearly one-hot
for d in [8, 64, 512]:
    q, k = rng.normal(size=(1000, d)), rng.normal(size=(1000, 10, d))
    raw = np.einsum("nd,nkd->nk", q, k)
    top_raw = softmax(raw).max(axis=1).mean()
    top_scaled = softmax(raw / np.sqrt(d)).max(axis=1).mean()
    print(f"d {d:3d}: spread of dot products {raw.std():6.1f}   average top weight, unscaled {top_raw:.2f}, scaled {top_scaled:.2f}")

d   8: spread of dot products    2.9   average top weight, unscaled 0.62, scaled 0.31
d  64: spread of dot products    8.0   average top weight, unscaled 0.85, scaled 0.31
d 512: spread of dot products   22.5   average top weight, unscaled 0.95, scaled 0.32


In [4]:
# Residual connections: gradient reaching the first of 50 layers, with and without a shortcut h + f(h)
def first_layer_grad(residual, depth=50, width=64):
    W = [rng.normal(0, 0.5 / np.sqrt(width), size=(width, width)) for _ in range(depth)]   # small weights
    h = rng.normal(size=(32, width)); zs = []
    for Wl in W:
        z = h @ Wl; zs.append(z)
        h = h + np.tanh(z) if residual else np.tanh(z)
    g = np.ones_like(h)
    for Wl, z in zip(reversed(W), reversed(zs)):
        local = (g * (1 - np.tanh(z) ** 2)) @ Wl.T              # gradient through the layer itself
        g = g + local if residual else local                    # the shortcut adds an identity path
    return np.linalg.norm(g)

print("plain 50-layer network, gradient at layer 1:   ", f"{first_layer_grad(False):.2e}")
print("residual 50-layer network, gradient at layer 1:", f"{first_layer_grad(True):.2e}")

plain 50-layer network, gradient at layer 1:    2.04e-14
residual 50-layer network, gradient at layer 1: 4.02e+02


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [5]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
# A Transformer would read the listing text word by word; here we just look at what it would see.
for d in df["description"].head(3):
    tokens = d.split()
    print(len(tokens), "tokens:", tokens)
print("vocabulary size:", len({w for d in df["description"] for w in d.split()}))

5 tokens: ['bright', 'compact', 'flat', 'near', 'metro']
6 tokens: ['spacious', 'flat', 'near', 'metro', 'with', 'balcony']
5 tokens: ['large', 'family', 'flat', 'needs', 'renovation']
vocabulary size: 23


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [6]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: Convolution output size

Compute the output width for input 32, kernel 5, padding 2, stride 1.

In [7]:
n, k, p, s = 32, 5, 2, 1
out = None  # ← your code here
check(out, 32, hint='(n − k + 2p) / s + 1.')

❌ Not yet. Hint: (n − k + 2p) / s + 1.


### Challenge 2: Softmax

Turn the attention scores into weights that sum to 1.

In [8]:
import numpy as np
scores = np.array([2.0, 0.0, 1.0])
weights = None  # ← your code here
check(weights, [0.6652409557748219, 0.09003057317038046, 0.24472847105479764], hint='e^score divided by the sum of all e^score.')

❌ Not yet. Hint: e^score divided by the sum of all e^score.


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. Convolution output size** — the missing line:

```python
out = (n - k + 2 * p) // s + 1
```

**2. Softmax** — the missing line:

```python
weights = np.exp(scores) / np.exp(scores).sum()
```